In [13]:
import polars as pl
import shap
import matplotlib as plt

In [14]:
version ='6.0.0'

In [16]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import polars as pl
import shap

# ==============================================================================
# 1. CONFIGURATION DE RÉFÉRENCE & CONFIGURATION GLOBALE
# ==============================================================================
plt.ioff()  # Empêche matplotlib d'ouvrir des fenêtres graphiques en boucle en RAM

# Définition des années et types d'élections à parser
PRES_YEARS = [2002, 2007, 2012, 2017, 2022]
LEG_YEARS = [2002, 2007, 2012, 2017, 2022, 2024]

# Mapping direct O(1) pour remplacer vos conditions "if var in ..."

BLOC_MAPPING = {
    "pvoteTG": "['tauTD', 'tauTG', 'taupar']",
    "pvoteTD": "['tauTD', 'tauTG', 'taupar']",
    "pvotepar":"['tauTD', 'tauTG', 'taupar']",
    "pvoteG": "['tauC', 'tauCD', 'tauCG', 'tauD', 'tauG', 'taupar']",
    "pvoteCG": "['tauC', 'tauCD', 'tauCG', 'tauD', 'tauG', 'taupar']",
    "pvoteCD": "['tauC', 'tauCD', 'tauCG', 'tauD', 'tauG', 'taupar']",
    "pvoteC": "['tauC', 'tauCD', 'tauCG', 'tauD', 'tauG', 'taupar']",
    "pvoteD": "['tauC', 'tauCD', 'tauCG', 'tauD', 'tauG', 'taupar']",
    "pvoteCGCCD": "['tauCGCCD', 'tauG', 'tauD', 'taupar]",
}



VARIABLES_DE_BOUCLE = list(BLOC_MAPPING.keys())

# Chemin constant du dataset global (chargé de manière Lazy une seule fois pour tout le script)
DATA_PARQUET_S3 = "s3://arthurmanceau/election_modeling_uhcp/data/derived/processed/data_processed_presidentiel_legislative_from1800_to2027_20260707_143756.parquet/"
STORAGE_OPTIONS = {"aws_skip_signature": "true"}

lazy_global_data = pl.scan_parquet(DATA_PARQUET_S3, storage_options=STORAGE_OPTIONS)

# ==============================================================================
# 2. PIPELINE PRINCIPAL D'EXÉCUTION
# ==============================================================================
# Itération conjointe sur le couple (type d'élection, liste des années associées)
for election_type, years in [("pres", PRES_YEARS), ("leg", LEG_YEARS)]:
    
    # Traduction pour le filtrage dans votre parquet global
    filter_election_str = "presidentiel" if election_type == "pres" else "legislative"
    
    for year in years:
        for var in VARIABLES_DE_BOUCLE:
            
            # Récupération instantanée du bloc de tendances associé
            trends = BLOC_MAPPING[var]
            
            # --- Étape A : Chargement de shap_df ---
            shap_path = f"s3://arthurmanceau/election_modeling_uhcp/data/output/explain/shap_values_{trends}_{var.replace('pvote', '')}_{year}_{election_type}_{version}.parquet"
            
            try:
                shap_df = pl.scan_parquet(shap_path, storage_options=STORAGE_OPTIONS, glob=False).collect()
            except Exception as e:
                print(f"Fichier SHAP introuvable ou invalide : {shap_path}. Passage à l'itération suivante.")
                continue

            features = sorted(list(set(shap_df.columns) - {"codecommune", "base_value"}))
            
            # --- Étape B : Extraction et Alignement Strict des Données ---
            # Le left-join garantit le même ordre de lignes (Communes) que shap_df
            data_ordered = (
                shap_df.lazy()
                .select("codecommune")
                .join(
                    lazy_global_data.filter(
                        (pl.col("election_type") == filter_election_str) & 
                        (pl.col("annee") == int(year))
                    ).select(features + ["codecommune"]),
                    on="codecommune",
                    how="left"
                )
                .collect()
            )
            
            # --- Étape C : Matrice NumPy & Création de l'Objet Explanation ---
            explanation = shap.Explanation(
                values=shap_df.select(features).to_pandas().to_numpy(),
                data=data_ordered.select(features).to_pandas().to_numpy(),
                feature_names=features,
            )
            
            # --- Étape D : Itération Graphique & Sauvegarde Économe en Mémoire ---
            output_dir = f"shap_scatter_plots/shap_plots_{election_type}_{year}_{var}"
            os.makedirs(output_dir, exist_ok=True)
            
            num_features = len(features)
            for i in range(num_features):
                #for j in range(num_features):
                
                    feature_x = features[i]
                    #feature_y = features[j]
                    
                    try:
                        fig, ax = plt.subplots(figsize=(7, 5))
                        
                        shap.plots.scatter(
                            explanation[:, i],
                            color=explanation,
                            ax=ax,
                            show=False
                        )
                        
                        ax.tick_params(axis="both", labelsize=9)
                        plt.tight_layout()
                        
                        filename = f"scatter_{feature_x}.png".replace("/", "_")
                        plt.savefig(os.path.join(output_dir, filename), dpi=130, bbox_inches="tight")
                        
                    except Exception as img_err:
                        print(f"Erreur Image [{feature_x}] : {img_err}")
                    finally:
                        plt.close(fig) # Désallocation immédiate pour éviter le crash mémoire RAM

print("Pipeline global exécuté avec succès !")


KeyboardInterrupt: 